In [1]:
from neo4j import GraphDatabase
import pickle

In [2]:
with open("graph_lematized_data.pkl", "rb") as f:
    graph_lematized_data = pickle.load(f)

/Users/robertplanas/Documents/GitHub/kg-augmented-multimodal-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
relationship = graph_lematized_data[
    "ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446"
][0]

In [4]:
relationship

Relationship(head='analysis', head_type='work_of_art', relation='PROVIDE_QUALITATIVE_UNDERSTANDING_OF', tail='model classification error', tail_type='event', confidence=1.0, context='Los resultados del análisis anterior proporcionan una comprensión cualitativa de los errores de clasificación del modelo.', language='en')

In [5]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "123456789")

In [ ]:
from nltk import data


def add_graph_data(relationship):
    # The 'with' block ensures the driver closes correctly
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        # Check if the connection is alive
        driver.verify_connectivity()

        # 2. Define the Cypher query with parameters
        # MERGE is better than CREATE for scripts because it prevents duplicates
        query = f"""
        MERGE (h:`{relationship["head_type"]}` {{name: $head_name}})
        MERGE (t:`{relationship["tail_type"]}` {{name: $tail_name}})
        MERGE (h)-[r:`{relationship["relation"]}`]->(t)
        SET r.confidence = $conf
        RETURN h.name, type(r), t.name
        """

        # 3. Execute the query
        records, summary, keys = driver.execute_query(
            query,
            head_name=relationship["head_name"],
            tail_name=relationship["tail_name"],
            conf=relationship["confidence"],
            database_="neo4j",
        )

        # 4. Print results
        for record in records:
            print(f"Success: {record['result']}")